# Analisis Sentimen Ulasan Produk Tokopedia 2025

Notebook mencakup audit data, preprocessing teks bahasa Indonesia, Logistic Regression,
penanganan ketidakseimbangan kelas, serta eksperimen IndoBERT.

Semua model memakai satu held-out test set. Balancing, augmentasi, TF-IDF, dan SMOTE
hanya dipelajari dari train set untuk mencegah data leakage.

## 1. Persiapan lingkungan

In [ ]:
%pip install -q kagglehub==0.3.13 Sastrawi==1.0.1 wordcloud==1.9.4 imbalanced-learn==0.14.0 transformers==4.57.1 datasets==4.4.1 accelerate==1.11.0

In [ ]:
import hashlib
import importlib.metadata as metadata
import os
import random
import re
from pathlib import Path
from time import perf_counter

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from wordcloud import WordCloud

RANDOM_STATE = 42
LABELS = ["negative", "neutral", "positive"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

PACKAGE_NAMES = [
    "kagglehub",
    "Sastrawi",
    "wordcloud",
    "imbalanced-learn",
    "transformers",
    "datasets",
    "accelerate",
    "numpy",
    "pandas",
    "scikit-learn",
    "torch",
]
display(
    pd.Series(
        {package: metadata.version(package) for package in PACKAGE_NAMES},
        name="version",
    ).to_frame()
)

## 2. Memuat dataset

In [ ]:
dataset_dir = Path(kagglehub.dataset_download("salmanabdu/tokopedia-product-reviews-2025"))
csv_paths = sorted(dataset_dir.glob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"CSV tidak ditemukan di {dataset_dir}")

csv_path = csv_paths[0]
with csv_path.open("rb") as dataset_file:
    dataset_sha256 = hashlib.file_digest(dataset_file, "sha256").hexdigest()
df = pd.read_csv(csv_path)

required_columns = {"review_text", "sentiment_label", "rating"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

raw_rows = len(df)
df = df.dropna(subset=["review_text", "sentiment_label"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment_label"] = df["sentiment_label"].astype(str).str.lower().str.strip()
df = df[df["review_text"].ne("") & df["sentiment_label"].isin(LABELS)].copy()

review_key = df["review_text"].str.lower().str.replace(r"\s+", " ", regex=True)
duplicate_rows = int(review_key.duplicated().sum())
df = df.loc[~review_key.duplicated()].reset_index(drop=True)

group_candidates = ["product_id", "product_name", "product_url", "product_link"]
group_column = next(
    (
        column
        for column in group_candidates
        if column in df.columns and df[column].notna().all()
    ),
    None,
)

print(f"Dataset: {csv_path.name}")
print(f"SHA-256: {dataset_sha256}")
print(f"Baris mentah: {raw_rows:,}")
print(f"Duplikat review dihapus: {duplicate_rows:,}")
print(f"Baris bersih: {len(df):,}")
print(f"Kolom grup: {group_column or 'tidak tersedia'}")

## 3. Audit dan eksplorasi data (EDA)

In [ ]:
df.info()
display(df.describe(include="all"))
display(df.isna().sum().rename("missing_values"))
display(df["rating"].value_counts().sort_index().rename("jumlah"))
display(df["sentiment_label"].value_counts().rename("jumlah"))

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    x="sentiment_label",
    order=LABELS,
    hue="sentiment_label",
    palette="viridis",
    legend=False,
)
plt.title("Distribusi Label Sentimen")
plt.xlabel("Sentimen")
plt.ylabel("Jumlah Ulasan")
plt.show()

### 3.1 Split data

Split dibuat sekali sebelum preprocessing dan balancing. Jika identitas produk tersedia,
produk sama tidak boleh tersebar ke train, validation, dan test set.

In [ ]:
def make_data_split(data):
    use_group_split = False
    if group_column:
        groups_per_label = data.groupby("sentiment_label")[group_column].nunique()
        use_group_split = len(groups_per_label) == len(LABELS) and groups_per_label.min() >= 5

    if use_group_split:
        outer_splitter = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        train_validation_indices, test_indices = next(
            outer_splitter.split(
                data,
                data["sentiment_label"],
                groups=data[group_column],
            )
        )
        train_validation = data.iloc[train_validation_indices]
        test = data.iloc[test_indices]

        inner_splitter = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        train_indices, validation_indices = next(
            inner_splitter.split(
                train_validation,
                train_validation["sentiment_label"],
                groups=train_validation[group_column],
            )
        )
        train = train_validation.iloc[train_indices]
        validation = train_validation.iloc[validation_indices]
    else:
        train_validation, test = train_test_split(
            data,
            test_size=0.2,
            random_state=RANDOM_STATE,
            stratify=data["sentiment_label"],
        )
        train, validation = train_test_split(
            train_validation,
            test_size=0.25,
            random_state=RANDOM_STATE,
            stratify=train_validation["sentiment_label"],
        )

    parts = tuple(part.reset_index(drop=True) for part in (train, validation, test))
    return (*parts, use_group_split)


train_data, validation_data, test_data, used_group_split = make_data_split(df)

for left, right in [
    (train_data, validation_data),
    (train_data, test_data),
    (validation_data, test_data),
]:
    assert set(left["review_text"]).isdisjoint(right["review_text"])
    if used_group_split:
        assert set(left[group_column]).isdisjoint(right[group_column])

print(f"Strategi split: {'stratified group' if used_group_split else 'stratified'}")
split_summary = pd.DataFrame(
    {
        "jumlah": [len(train_data), len(validation_data), len(test_data)],
        "proporsi": [len(train_data), len(validation_data), len(test_data)],
    },
    index=["train", "validation", "test"],
)
split_summary["proporsi"] /= len(df)
display(split_summary.style.format({"proporsi": "{:.1%}"}))

label_distribution = pd.concat(
    {
        "train": train_data["sentiment_label"].value_counts(normalize=True),
        "validation": validation_data["sentiment_label"].value_counts(normalize=True),
        "test": test_data["sentiment_label"].value_counts(normalize=True),
    },
    axis=1,
).reindex(LABELS)
display(label_distribution.style.format("{:.1%}"))

## 4. Preprocessing teks

In [ ]:
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return stopword_remover.remove(text)

sample_text = df["review_text"].dropna().iloc[0]
print("Sebelum:", sample_text)
print("Sesudah:", clean_text(sample_text))

In [ ]:
stemmer = StemmerFactory().create_stemmer()

def clean_text_advanced(text):
    return stemmer.stem(clean_text(text))

print(f"Sebelum stemming: {sample_text[:50]}...")
print(f"Sesudah stemming: {clean_text_advanced(sample_text)}")


### 4.1 Normalisasi slang dan emotikon

Bagian ini menambahkan kamus slang dan konversi emotikon untuk meningkatkan kualitas fitur teks.


In [ ]:
slang_dict = {
    "bgt": "banget", "bgtt": "banget", "jd": "jadi", "ga": "tidak",
    "gak": "tidak", "tdk": "tidak", "sdh": "sudah", "udah": "sudah",
    "brg": "barang", "krn": "karena", "cpt": "cepat", "jg": "juga",
    "tp": "tapi", "kalo": "kalau", "dgn": "dengan", "dr": "dari",
    "ori": "original", "mksh": "terima kasih", "tks": "terima kasih"
}

emoticon_map = {
    "😊": " senang ", "🙂": " senang ", "😄": " senang ", "😍": " suka ",
    "👍": " bagus ", "👌": " bagus ", "⭐": " bintang ",
    "😞": " kecewa ", "😟": " kecewa ", "😠": " marah ", "😡": " marah ",
    "👎": " buruk ", "😢": " sedih ", "😭": " sedih "
}

def normalize_slang(text):
    words = text.split()
    return " ".join([slang_dict.get(w, w) for w in words])

def convert_emoticons(text):
    for emo, replacement in emoticon_map.items():
        text = text.replace(emo, replacement)
    return text

def clean_text_final(text):
    # Konversi emotikon sebelum regex menghapusnya
    text = convert_emoticons(str(text))
    text = clean_text(text)
    text = normalize_slang(text)
    return stemmer.stem(text)

test_slang = "barangnya bgt bagus, tp pengiriman agak lama jd kecewa 😊"
print(f"Original: {test_slang}")
print(f"Processed: {clean_text_final(test_slang)}")

## 5. Pelatihan model baseline

In [ ]:
X_train = train_data["review_text"].apply(clean_text)
X_validation = validation_data["review_text"].apply(clean_text)
X_test = test_data["review_text"].apply(clean_text)
y_train = train_data["sentiment_label"]
y_validation = validation_data["sentiment_label"]
y_test = test_data["sentiment_label"]

baseline_tfidf = TfidfVectorizer(max_features=2_000)
X_train_tfidf = baseline_tfidf.fit_transform(X_train)
X_validation_tfidf = baseline_tfidf.transform(X_validation)
X_test_tfidf = baseline_tfidf.transform(X_test)

baseline_model = LogisticRegression(
    max_iter=1_000,
    random_state=RANDOM_STATE,
)
started_at = perf_counter()
baseline_model.fit(X_train_tfidf, y_train)
baseline_training_seconds = perf_counter() - started_at

## 6. Evaluasi model baseline

In [ ]:
experiment_results = []


def evaluate_predictions(
    model_name,
    y_true,
    y_pred,
    training_seconds,
    inference_seconds,
    validation_f1_macro,
):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )
    _, recall_per_class, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        zero_division=0,
    )
    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "validation_f1_macro": validation_f1_macro,
        "neutral_recall": recall_per_class[LABELS.index("neutral")],
        "training_seconds": training_seconds,
        "inference_seconds": inference_seconds,
    }
    experiment_results[:] = [
        row for row in experiment_results if row["model"] != model_name
    ]
    experiment_results.append(result)
    print(f"{model_name} | macro F1: {f1:.4f}")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    return result


def macro_f1(y_true, y_pred):
    return precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )[2]


def plot_confusion(y_true, y_pred, title, cmap="Blues"):
    matrix = confusion_matrix(y_true, y_pred, labels=LABELS)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap=cmap,
        xticklabels=LABELS,
        yticklabels=LABELS,
    )
    plt.xlabel("Prediksi")
    plt.ylabel("Aktual")
    plt.title(title)
    plt.show()


y_pred_baseline_validation = baseline_model.predict(X_validation_tfidf)
baseline_validation_f1 = macro_f1(y_validation, y_pred_baseline_validation)

started_at = perf_counter()
y_pred_baseline = baseline_model.predict(X_test_tfidf)
baseline_inference_seconds = perf_counter() - started_at
evaluate_predictions(
    "Baseline Logistic Regression",
    y_test,
    y_pred_baseline,
    baseline_training_seconds,
    baseline_inference_seconds,
    baseline_validation_f1,
)
plot_confusion(y_test, y_pred_baseline, "Confusion Matrix: Baseline")

## 7. Optimasi hyperparameter

In [ ]:
X_train_advanced = train_data["review_text"].apply(clean_text_advanced)
X_validation_advanced = validation_data["review_text"].apply(clean_text_advanced)
X_test_advanced = test_data["review_text"].apply(clean_text_advanced)

pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]
)
param_grid = {
    "tfidf__max_features": [1_000, 2_000, 3_000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C": [0.1, 1, 10],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring="f1_macro",
)
started_at = perf_counter()
grid_search.fit(X_train_advanced, y_train)
tuned_training_seconds = perf_counter() - started_at

print(f"Parameter terbaik: {grid_search.best_params_}")
print(f"Skor CV terbaik: {grid_search.best_score_:.4f}")

In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned_validation = best_model.predict(X_validation_advanced)
tuned_validation_f1 = macro_f1(y_validation, y_pred_tuned_validation)

started_at = perf_counter()
y_pred_tuned = best_model.predict(X_test_advanced)
tuned_inference_seconds = perf_counter() - started_at

evaluate_predictions(
    "Tuned Logistic Regression",
    y_test,
    y_pred_tuned,
    tuned_training_seconds,
    tuned_inference_seconds,
    tuned_validation_f1,
)
plot_confusion(y_test, y_pred_tuned, "Confusion Matrix: Tuned Model", "Purples")

## 8. Visualisasi kata dominan

In [ ]:
wordcloud_data = train_data.assign(
    clean_text=train_data["review_text"].apply(clean_text_advanced)
)


def show_wordcloud(sentiment, colormap):
    text = " ".join(
        wordcloud_data.loc[
            wordcloud_data["sentiment_label"] == sentiment,
            "clean_text",
        ]
    )
    image = WordCloud(
        width=800,
        height=400,
        background_color="white",
        colormap=colormap,
        max_words=100,
    ).generate(text)

    plt.figure(figsize=(10, 5))
    plt.imshow(image, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Kata Dominan: {sentiment.upper()}")
    plt.show()


for sentiment, colormap in zip(LABELS, ["Reds", "plasma", "viridis"]):
    show_wordcloud(sentiment, colormap)

## 9. Penanganan ketidakseimbangan kelas

Semua strategi hanya mengubah train set. Held-out test set tetap sama.

### 9.1 Augmentasi teks

In [ ]:
synonym_dict = {
    "bagus": ["baik", "oke", "mantap", "keren"],
    "buruk": ["jelek", "kurang", "parah"],
    "kecewa": ["kesal", "nyesel", "sedih"],
    "cepat": ["kilat", "gegas", "ekspres"],
    "lama": ["lambat", "lelet", "telat"],
}


def augment_text(text):
    words = text.split()
    return " ".join(
        random.choice(synonym_dict[word])
        if word in synonym_dict and random.random() > 0.5
        else word
        for word in words
    )


class_counts = train_data["sentiment_label"].value_counts()
target_count = int(class_counts.max())
augmented_parts = [train_data]
for label in LABELS:
    label_rows = train_data[train_data["sentiment_label"] == label]
    required_rows = target_count - len(label_rows)
    if required_rows <= 0:
        continue
    synthetic_rows = label_rows.sample(
        n=required_rows,
        replace=required_rows > len(label_rows),
        random_state=RANDOM_STATE,
    ).copy()
    synthetic_rows["review_text"] = (
        synthetic_rows["review_text"].apply(clean_text).apply(augment_text)
    )
    augmented_parts.append(synthetic_rows)

augmented_train = pd.concat(augmented_parts, ignore_index=True)
augmented_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(max_features=3_000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]
)
started_at = perf_counter()
augmented_model.fit(
    augmented_train["review_text"].apply(clean_text_final),
    augmented_train["sentiment_label"],
)
augmented_training_seconds = perf_counter() - started_at

y_pred_augmented_validation = augmented_model.predict(
    validation_data["review_text"].apply(clean_text_final)
)
augmented_validation_f1 = macro_f1(y_validation, y_pred_augmented_validation)

started_at = perf_counter()
y_pred_augmented = augmented_model.predict(
    test_data["review_text"].apply(clean_text_final)
)
augmented_inference_seconds = perf_counter() - started_at
evaluate_predictions(
    "Augmented Logistic Regression",
    y_test,
    y_pred_augmented,
    augmented_training_seconds,
    augmented_inference_seconds,
    augmented_validation_f1,
)

### 9.2 SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

X_train_smote_text = train_data["review_text"].apply(clean_text_final)
X_validation_smote_text = validation_data["review_text"].apply(clean_text_final)
X_test_smote_text = test_data["review_text"].apply(clean_text_final)

tfidf_smote = TfidfVectorizer(max_features=2_000)
X_train_smote = tfidf_smote.fit_transform(X_train_smote_text)
X_validation_smote = tfidf_smote.transform(X_validation_smote_text)
X_test_smote = tfidf_smote.transform(X_test_smote_text)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_smote, y_train)

print("Train sebelum SMOTE:", y_train.value_counts().to_dict())
print("Train setelah SMOTE:", pd.Series(y_train_resampled).value_counts().to_dict())

In [ ]:
smote_model = LogisticRegression(
    max_iter=1_000,
    random_state=RANDOM_STATE,
)
started_at = perf_counter()
smote_model.fit(X_train_resampled, y_train_resampled)
smote_training_seconds = perf_counter() - started_at

y_pred_smote_validation = smote_model.predict(X_validation_smote)
smote_validation_f1 = macro_f1(y_validation, y_pred_smote_validation)

started_at = perf_counter()
y_pred_smote = smote_model.predict(X_test_smote)
smote_inference_seconds = perf_counter() - started_at
evaluate_predictions(
    "SMOTE Logistic Regression",
    y_test,
    y_pred_smote,
    smote_training_seconds,
    smote_inference_seconds,
    smote_validation_f1,
)

In [ ]:
plot_confusion(
    y_test,
    y_pred_smote,
    "Confusion Matrix: SMOTE",
    "Greens",
)

### 9.3 Class weighting

In [ ]:
weighted_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(max_features=3_000)),
        (
            "clf",
            LogisticRegression(
                max_iter=1_000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
started_at = perf_counter()
weighted_model.fit(X_train, y_train)
weighted_training_seconds = perf_counter() - started_at

y_pred_weighted_validation = weighted_model.predict(X_validation)
weighted_validation_f1 = macro_f1(y_validation, y_pred_weighted_validation)

started_at = perf_counter()
y_pred_weighted = weighted_model.predict(X_test)
weighted_inference_seconds = perf_counter() - started_at
evaluate_predictions(
    "Weighted Logistic Regression",
    y_test,
    y_pred_weighted,
    weighted_training_seconds,
    weighted_inference_seconds,
    weighted_validation_f1,
)
plot_confusion(
    y_test,
    y_pred_weighted,
    "Confusion Matrix: Class Weighting",
    "Oranges",
)

### 9.4 Peningkatan deteksi kelas netral

Eksperimen menggabungkan fitur panjang teks, train set seimbang, dan penyesuaian
probabilitas untuk memperkuat sinyal kelas `neutral`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion


n_balanced = train_data["sentiment_label"].value_counts().min()
balanced_train = pd.concat(
    [
        train_data[train_data["sentiment_label"] == label].sample(
            n=n_balanced,
            random_state=RANDOM_STATE,
        )
        for label in LABELS
    ],
    ignore_index=True,
)

X_train_balanced = balanced_train["review_text"].apply(clean_text)
y_train_balanced = balanced_train["sentiment_label"]


class TextStats(BaseEstimator, TransformerMixin):
    def fit(self, texts, labels=None):
        return self

    def transform(self, texts):
        return np.array([len(str(text)) for text in texts]).reshape(-1, 1)


enhanced_pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    (
                        "tfidf",
                        TfidfVectorizer(max_features=3_000, ngram_range=(1, 2)),
                    ),
                    ("stats", TextStats()),
                ]
            ),
        ),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
started_at = perf_counter()
enhanced_pipeline.fit(X_train_balanced, y_train_balanced)
enhanced_training_seconds = perf_counter() - started_at

X_validation_enhanced = validation_data["review_text"].apply(clean_text)
validation_probabilities = enhanced_pipeline.predict_proba(X_validation_enhanced)
neutral_index = list(enhanced_pipeline.classes_).index("neutral")

neutral_bonus_candidates = np.arange(0.0, 0.31, 0.05)
validation_scores = []
for bonus in neutral_bonus_candidates:
    adjusted_probabilities = validation_probabilities.copy()
    adjusted_probabilities[:, neutral_index] += bonus
    validation_predictions = enhanced_pipeline.classes_[
        np.argmax(adjusted_probabilities, axis=1)
    ]
    _, _, macro_f1, _ = precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )
    validation_scores.append(macro_f1)

neutral_bonus = float(neutral_bonus_candidates[np.argmax(validation_scores)])
print(f"Bonus probabilitas neutral terpilih: {neutral_bonus:.2f}")

X_test_enhanced = test_data["review_text"].apply(clean_text)
started_at = perf_counter()
y_probabilities = enhanced_pipeline.predict_proba(X_test_enhanced)
y_probabilities[:, neutral_index] += neutral_bonus
y_pred_enhanced = enhanced_pipeline.classes_[np.argmax(y_probabilities, axis=1)]
enhanced_inference_seconds = perf_counter() - started_at

In [ ]:
evaluate_predictions(
    "Enhanced Logistic Regression",
    y_test,
    y_pred_enhanced,
    enhanced_training_seconds,
    enhanced_inference_seconds,
    max(validation_scores),
)
plot_confusion(
    y_test,
    y_pred_enhanced,
    "Confusion Matrix: Enhanced Model",
    "Purples",
)

In [ ]:
positive_as_neutral_mask = (
    (y_test.to_numpy() == "positive")
    & (np.asarray(y_pred_enhanced) == "neutral")
)
positive_as_neutral = test_data.loc[
    positive_as_neutral_mask,
    ["review_text", "sentiment_label"],
].reset_index(drop=True)

print(f"Total positive diprediksi neutral: {len(positive_as_neutral)}")
display(positive_as_neutral.head(10))

## 10. Eksperimen IndoBERT

Bagian opsional ini memakai `indobenchmark/indobert-base-p2`. Jalankan dengan runtime
GPU agar fine-tuning lebih cepat.

In [ ]:
from collections import Counter
from glob import glob

import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

BERT_MODEL_NAME = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for index, label in enumerate(LABELS)}

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)


def tokenize_function(examples):
    return tokenizer(
        examples["review_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

### 10.1 Menyiapkan dataset

In [ ]:
train_dataset = Dataset.from_pandas(
    train_data[["review_text", "sentiment_label"]],
    preserve_index=False,
)
validation_dataset = Dataset.from_pandas(
    validation_data[["review_text", "sentiment_label"]],
    preserve_index=False,
)
test_dataset = Dataset.from_pandas(
    test_data[["review_text", "sentiment_label"]],
    preserve_index=False,
)


def add_label(row):
    return {"labels": label2id[row["sentiment_label"]]}


train_dataset = train_dataset.map(add_label).map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(add_label).map(
    tokenize_function,
    batched=True,
)
test_dataset = test_dataset.map(add_label).map(tokenize_function, batched=True)

### 10.2 Fine-tuning

In [ ]:
def compute_metrics(prediction):
    labels = prediction.label_ids
    predictions = prediction.predictions.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


training_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=10,
    report_to="none",
    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
)
started_at = perf_counter()
trainer.train()
bert_training_seconds = perf_counter() - started_at

### 10.3 Evaluasi IndoBERT

In [ ]:
validation_output = trainer.predict(validation_dataset)
y_pred_bert_validation = np.array(
    [id2label[index] for index in validation_output.predictions.argmax(axis=-1)]
)
y_true_bert_validation = np.array(
    [id2label[index] for index in validation_output.label_ids]
)
bert_validation_f1 = macro_f1(y_true_bert_validation, y_pred_bert_validation)

started_at = perf_counter()
prediction_output = trainer.predict(test_dataset)
bert_inference_seconds = perf_counter() - started_at
y_pred_bert_ids = np.argmax(prediction_output.predictions, axis=-1)
y_true_bert_ids = prediction_output.label_ids
y_pred_bert = np.array([id2label[index] for index in y_pred_bert_ids])
y_true_bert = np.array([id2label[index] for index in y_true_bert_ids])

evaluate_predictions(
    "IndoBERT",
    y_true_bert,
    y_pred_bert,
    bert_training_seconds,
    bert_inference_seconds,
    bert_validation_f1,
)

In [ ]:
plot_confusion(
    y_true_bert,
    y_pred_bert,
    "Confusion Matrix: IndoBERT",
    "Oranges",
)

### 10.4 Analisis kesalahan

In [ ]:
error_indices = np.where(y_pred_bert != y_true_bert)[0]
error_samples = pd.DataFrame(
    {
        "Ulasan": test_data.iloc[error_indices[:10]]["review_text"].to_numpy(),
        "Aktual": y_true_bert[error_indices[:10]],
        "Prediksi": y_pred_bert[error_indices[:10]],
    }
)
display(error_samples)

### 10.5 Visualisasi bobot attention

In [ ]:
checkpoints = glob("results/checkpoint-*")
last_checkpoint = (
    max(checkpoints, key=os.path.getctime) if checkpoints else BERT_MODEL_NAME
)
print(f"Memuat model dari: {last_checkpoint}")

attention_model = AutoModelForSequenceClassification.from_pretrained(
    last_checkpoint,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    attn_implementation="eager",
).to(trainer.args.device)


def visualize_attention(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(attention_model.device)

    with torch.no_grad():
        outputs = attention_model(**inputs, output_attentions=True)

    attention = outputs.attentions[-1][0, 0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        attention[: len(tokens), : len(tokens)],
        xticklabels=tokens,
        yticklabels=tokens,
        cmap="YlGnBu",
    )
    plt.title(f"Attention Map\nUlasan: {text[:50]!r}")
    plt.xticks(rotation=45)
    plt.show()


sample_review = "barangnya rusak parah dan pengiriman sangat lambat"
visualize_attention(sample_review)


### 10.6 Perbandingan model

Semua skor berasal dari held-out test set yang sama.

In [ ]:
comparison_df = (
    pd.DataFrame(experiment_results)
    .set_index("model")
    .sort_values("validation_f1_macro", ascending=False)
)

metric_columns = [
    "validation_f1_macro",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "neutral_recall",
]
display(
    comparison_df.style.format(
        {
            **{column: "{:.2%}" for column in metric_columns},
            "training_seconds": "{:.2f}",
            "inference_seconds": "{:.4f}",
        }
    )
)

final_model_name = comparison_df["validation_f1_macro"].idxmax()
print(f"Model final berdasarkan validation macro F1: {final_model_name}")

### 10.7 Pengujian IndoBERT pada salah klasifikasi positif sebagai netral

IndoBERT diuji pada review positif di held-out test set yang diprediksi sebagai kelas
`neutral` oleh Enhanced Logistic Regression.

In [ ]:
misclassified_texts = positive_as_neutral["review_text"].tolist()

if not misclassified_texts:
    print("Tidak ada kasus positive-as-neutral untuk diuji.")
else:
    misclassified_dataset = Dataset.from_dict({"review_text": misclassified_texts})

    def tokenize_test(examples):
        return tokenizer(
            examples["review_text"],
            padding="max_length",
            truncation=True,
            max_length=128,
        )

    misclassified_tokenized = misclassified_dataset.map(tokenize_test, batched=True)
    bert_predictions = trainer.predict(misclassified_tokenized)
    y_pred_bert_indices = np.argmax(bert_predictions.predictions, axis=-1)
    y_pred_bert_labels = [id2label[index] for index in y_pred_bert_indices]

    counts = Counter(y_pred_bert_labels)
    print(f"Total sampel diuji: {len(misclassified_texts)}")
    print("Hasil prediksi IndoBERT:")
    for label, count in counts.items():
        print(f"- {label}: {count} ({count / len(misclassified_texts):.2%})")

    print("\nContoh ulasan yang diperbaiki IndoBERT:")
    correct_indices = [
        index
        for index, label in enumerate(y_pred_bert_labels)
        if label == "positive"
    ][:5]
    for index in correct_indices:
        print(f"Ulasan: {misclassified_texts[index]}")

## 11. Prediksi ulasan baru

Fungsi memakai model dengan macro F1 tertinggi pada held-out test set.

In [ ]:
def predict_enhanced(text):
    probabilities = enhanced_pipeline.predict_proba([clean_text(text)])
    probabilities[:, neutral_index] += neutral_bonus
    class_index = probabilities.argmax(axis=1)[0]
    return enhanced_pipeline.classes_[class_index]


def predict_indobert(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(trainer.model.device)
    trainer.model.eval()
    with torch.no_grad():
        logits = trainer.model(**inputs).logits
    return id2label[int(logits.argmax(axis=-1).item())]


model_predictors = {
    "Baseline Logistic Regression": lambda text: baseline_model.predict(
        baseline_tfidf.transform([clean_text(text)])
    )[0],
    "Tuned Logistic Regression": lambda text: best_model.predict(
        [clean_text_advanced(text)]
    )[0],
    "Augmented Logistic Regression": lambda text: augmented_model.predict(
        [clean_text_final(text)]
    )[0],
    "SMOTE Logistic Regression": lambda text: smote_model.predict(
        tfidf_smote.transform([clean_text_final(text)])
    )[0],
    "Weighted Logistic Regression": lambda text: weighted_model.predict(
        [clean_text(text)]
    )[0],
    "Enhanced Logistic Regression": predict_enhanced,
    "IndoBERT": predict_indobert,
}


def prediksi_sentimen(teks):
    return model_predictors[final_model_name](teks)


contoh_ulasan = [
    "Produknya bagus banget dan pengiriman cepat",
    "Barang rusak dan penjual tidak responsif",
    "Produknya cukup, tidak buruk tapi juga tidak istimewa",
]
for ulasan in contoh_ulasan:
    print(f"{ulasan!r} -> {prediksi_sentimen(ulasan)}")

## 12. Kesimpulan

Pemilihan model memakai macro F1 agar seluruh kelas mendapat bobot setara. Recall kelas
`neutral` dilaporkan terpisah karena kelas ini menjadi fokus kesalahan utama.

In [ ]:
winner = comparison_df.loc[final_model_name]
print(f"Model terbaik: {final_model_name}")
print(f"Validation macro F1: {winner['validation_f1_macro']:.2%}")
print(f"Held-out test macro F1: {winner['f1_macro']:.2%}")
print(f"Recall neutral: {winner['neutral_recall']:.2%}")
print(f"Waktu training: {winner['training_seconds']:.2f} detik")
print(f"Waktu inference test set: {winner['inference_seconds']:.4f} detik")

print("\nKeterbatasan:")
print("- Data berasal dari satu dataset Tokopedia 2025.")
print("- Normalisasi slang dan augmentasi masih berbasis kamus kecil.")
print("- Group split hanya aktif bila identitas produk tersedia dan memadai.")
print("- Kandidat threshold neutral masih dibatasi pada rentang 0.00-0.30.")

print("\nRekomendasi:")
print("- Tambah data lintas kategori dan periode.")
print("- Tuning threshold hanya pada validation set.")
print("- Uji stabilitas hasil pada beberapa random seed.")